In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

In [2]:
df = pd.read_csv('../../data/DieCasting_Quality_Raw_Data.csv',header=[0,1])
df

Process                                                     \
           id Product_Type Shot Velocity_1 Velocity_2 Velocity_3   
0           1            1    1      0.144      0.170      0.188   
1        1002            1    2      0.144      0.170      0.182   
2        2003            1    3      0.144      0.170      0.182   
3        3004            1    4      0.144      0.170      0.182   
4        4005            1    5      0.144      0.172      0.176   
...       ...          ...  ...        ...        ...        ...   
7530  7530659            2  659      0.150      0.166      0.210   
7531  7531660            2  660      0.144      0.174      0.206   
7532  7532660            2  660      0.144      0.174      0.206   
7533  7533661            2  661      0.147      0.174      0.204   
7534  7534661            2  661      0.147      0.174      0.204   

                                                                         ...  \
     High_Velocity Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness   ...   
0            2.134               214           0.008                 10  ...   
1            2.124               217           0.008                 11  ...   
2            2.116               214           0.008                 11  ...   
3            2.137               217           0.008                 11  ...   
4            2.111               217           0.008                 12  ...   
...            ...               ...             ...                ...  ...   
7530         2.492               265           0.011                 17  ...   
7531         2.514               264           0.011                 16  ...   
7532         2.514               264           0.011                 16  ...   
7533         2.532               265           0.012                 18  ...   
7534         2.532               265           0.012                 18  ...   

         Defects                                                          \
     Blow_Hole_2 Stain_2 Dent_2 Deformation_2 Contamination_2 Impurity_2   
0              0       0      0             0               0          0   
1              0       0      0             0               0          0   
2              0       0      0             0               0          0   
3              0       0      0             0               0          0   
4              0       0      0             0               0          0   
...          ...     ...    ...           ...             ...        ...   
7530           0       0      0             0               0          0   
7531           0       0      0             0               0          0   
7532           0       0      0             0               0          0   
7533           0       0      0             0               0          0   
7534           0       0      0             0               0          0   

                                                   
     Crack_2 Scratch_2 Buring_Mark_2 Inclusions_2  
0          0         0             0            0  
1          0         0             0            0  
2          0         0             0            0  
3          0         0             0            0  
4          0         0             0            0  
...      ...       ...           ...          ...  
7530       0         0             0            0  
7531       0         0             0            0  
7532       0         0             0            0  
7533       0         0             0            0  
7534       0         0             0            0  

[7535 rows x 57 columns]

1. 컬럼명 공백제거
2. 중복제거
3. 2번, 3번을 불량(1)로 대체
4. 새로운 파생컬럼 생성

In [3]:
### 1. 컬럼명 공백 제거
df.rename(columns={'Biscuit_Thickness ': 'Biscuit_Thickness', 'Clamping_Force ': 'Clamping_Force', ' Pressure_Rise_Time': 'Pressure_Rise_Time'}, level=1, inplace=True)

In [4]:
### 2. 중복 제거
# 제외할 컬럼 리스트 정의
exclude_cols = [('Process', 'id')]

# 전체 컬럼에서 제외할 컬럼만 뺀 리스트 생성
target_cols = df.columns.difference(exclude_cols).tolist()

# 중복 제거 실행 (첫 번째 행은 남기고 나머지는 삭제)
df_cleaned = df.drop_duplicates(subset=target_cols, keep='first').reset_index(drop=True)

# 결과 확인
print(f"제거 전 행 개수: {len(df)}")
print(f"제거 후 행 개수: {len(df_cleaned)}")
print(f"삭제된 중복 행 개수: {len(df) - len(df_cleaned)}")

제거 전 행 개수: 7535
제거 후 행 개수: 4617
삭제된 중복 행 개수: 2918


In [5]:
### 3. 2,3을 불량 1로 대체
defect_cols = df_cleaned['Defects'].columns

# 'Defects' 대분류 아래의 모든 컬럼에 대해 2 이상의 값을 1로 변경
for col in defect_cols:
    # Defects 하위의 col 값 중 2 이상인 경우를 1로 치환
    df_cleaned.loc[df_cleaned[('Defects', col)] >= 2, ('Defects', col)] = 1

In [6]:
### 4. 새로운 파생컬럼 생성
defect_cols = [col for col in df_cleaned.columns if col[0]=='Defects']

df_cleaned[('Defect_Flag','Is_Defect')] = (df_cleaned[defect_cols] == 1).any(axis=1).astype(int)

df_cleaned[('Defect_Flag','Is_Defect')].value_counts() # --> 0: 정상, 1: 불량

(Defect_Flag, Is_Defect)
0    3544
1    1073
Name: count, dtype: int64

In [ ]:
df_cleaned

In [7]:
### 5. 모든 컬럼명 소문자화
df_cleaned.columns = df_cleaned.columns.map(lambda x: tuple(str(level).lower() for level in x))

In [ ]:
df_cleaned

In [9]:
df_type_1 = df_cleaned[df_cleaned[('process', 'product_type')] == 1].reset_index(drop=True)

In [10]:
df_type_1

process                                                     \
           id product_type shot velocity_1 velocity_2 velocity_3   
0           1            1    1      0.144      0.170      0.188   
1        1002            1    2      0.144      0.170      0.182   
2        2003            1    3      0.144      0.170      0.182   
3        3004            1    4      0.144      0.170      0.182   
4        4005            1    5      0.144      0.172      0.176   
...       ...          ...  ...        ...        ...        ...   
2648  4197741            1  741      0.168      0.210      0.220   
2649  4199742            1  742      0.168      0.212      0.234   
2650  4201743            1  743      0.166      0.210      0.222   
2651  4203744            1  744      0.174      0.210      0.227   
2652  4205745            1  745      0.176      0.206      0.230   

                                                                        ...  \
     high_velocity cylinder_pressure rapid_rise_time biscuit_thickness  ...   
0            2.134               214           0.008                10  ...   
1            2.124               217           0.008                11  ...   
2            2.116               214           0.008                11  ...   
3            2.137               217           0.008                11  ...   
4            2.111               217           0.008                12  ...   
...            ...               ...             ...               ...  ...   
2648         2.217               239           0.009                12  ...   
2649         2.238               239           0.010                13  ...   
2650         2.277               239           0.008                12  ...   
2651         2.260               239           0.010                12  ...   
2652         2.254               239           0.007                12  ...   

     defects                                                          \
     stain_2 dent_2 deformation_2 contamination_2 impurity_2 crack_2   
0          0      0             0               0          0       0   
1          0      0             0               0          0       0   
2          0      0             0               0          0       0   
3          0      0             0               0          0       0   
4          0      0             0               0          0       0   
...      ...    ...           ...             ...        ...     ...   
2648       0      0             0               0          0       0   
2649       0      0             0               0          0       0   
2650       0      0             0               0          0       0   
2651       0      0             0               0          0       0   
2652       0      0             0               0          0       0   

                                          defect_flag  
     scratch_2 buring_mark_2 inclusions_2   is_defect  
0            0             0            0           0  
1            0             0            0           0  
2            0             0            0           0  
3            0             0            0           1  
4            0             0            0           0  
...        ...           ...          ...         ...  
2648         0             0            0           0  
2649         0             0            0           0  
2650         0             0            0           0  
2651         0             0            0           0  
2652         0             0            0           0  

[2653 rows x 58 columns]

In [12]:
defect_cols = [col for col in df_type_1.columns if 'defect' in str(col)]

for col in defect_cols:
    print(f"--- [Column: {col}] ---")
    print(df_type_1[col].value_counts())
    print("\n") 

# 각 컬럼의 value_counts를 데이터프레임 하나로 병합
counts_summary = df_type_1[defect_cols].apply(lambda x: x.value_counts()).fillna(0).astype(int)
display(counts_summary)

--- [Column: ('defects', 'short_shot_1')] ---
(defects, short_shot_1)
0    2545
1     108
Name: count, dtype: int64


--- [Column: ('defects', 'bubble_1')] ---
(defects, bubble_1)
0    2593
1      60
Name: count, dtype: int64


--- [Column: ('defects', 'exfoliation_1')] ---
(defects, exfoliation_1)
0    2529
1     124
Name: count, dtype: int64


--- [Column: ('defects', 'blow_hole_1')] ---
(defects, blow_hole_1)
0    2653
Name: count, dtype: int64


--- [Column: ('defects', 'stain_1')] ---
(defects, stain_1)
0    2650
1       3
Name: count, dtype: int64


--- [Column: ('defects', 'dent_1')] ---
(defects, dent_1)
0    2650
1       3
Name: count, dtype: int64


--- [Column: ('defects', 'deformation_1')] ---
(defects, deformation_1)
0    2545
1     108
Name: count, dtype: int64


--- [Column: ('defects', 'contamination_1')] ---
(defects, contamination_1)
0    2653
Name: count, dtype: int64


--- [Column: ('defects', 'impurity_1')] ---
(defects, impurity_1)
0    2653
Name: count, dtype: in

defects                                                    \
  short_shot_1 bubble_1 exfoliation_1 blow_hole_1 stain_1 dent_1   
0         2545     2593          2529        2653    2650   2650   
1          108       60           124           0       3      3   

                                                    ...                 \
  deformation_1 contamination_1 impurity_1 crack_1  ... stain_2 dent_2   
0          2545            2653       2653    2653  ...    2653   2653   
1           108               0          0       0  ...       0      0   

                                                                            \
  deformation_2 contamination_2 impurity_2 crack_2 scratch_2 buring_mark_2   
0          2589            2653       2653    2653      2653          2653   
1            64               0          0       0         0             0   

               defect_flag  
  inclusions_2   is_defect  
0         2653        2077  
1            0         576  

[2 rows x 27 columns]

In [13]:
# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (df_type_1[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

값이 0만 있는 컬럼들(16개): 
[('defects', 'blow_hole_1'), ('defects', 'contamination_1'), ('defects', 'impurity_1'), ('defects', 'crack_1'), ('defects', 'scratch_1'), ('defects', 'buring_mark_1'), ('defects', 'inclusions_1'), ('defects', 'blow_hole_2'), ('defects', 'stain_2'), ('defects', 'dent_2'), ('defects', 'contamination_2'), ('defects', 'impurity_2'), ('defects', 'crack_2'), ('defects', 'scratch_2'), ('defects', 'buring_mark_2'), ('defects', 'inclusions_2')] 


In [14]:
d_reduced = df_type_1.drop(columns=only_zero_cols)

d_reduced

process                                                     \
           id product_type shot velocity_1 velocity_2 velocity_3   
0           1            1    1      0.144      0.170      0.188   
1        1002            1    2      0.144      0.170      0.182   
2        2003            1    3      0.144      0.170      0.182   
3        3004            1    4      0.144      0.170      0.182   
4        4005            1    5      0.144      0.172      0.176   
...       ...          ...  ...        ...        ...        ...   
2648  4197741            1  741      0.168      0.210      0.220   
2649  4199742            1  742      0.168      0.212      0.234   
2650  4201743            1  743      0.166      0.210      0.222   
2651  4203744            1  744      0.174      0.210      0.227   
2652  4205745            1  745      0.176      0.206      0.230   

                                                                        ...  \
     high_velocity cylinder_pressure rapid_rise_time biscuit_thickness  ...   
0            2.134               214           0.008                10  ...   
1            2.124               217           0.008                11  ...   
2            2.116               214           0.008                11  ...   
3            2.137               217           0.008                11  ...   
4            2.111               217           0.008                12  ...   
...            ...               ...             ...               ...  ...   
2648         2.217               239           0.009                12  ...   
2649         2.238               239           0.010                13  ...   
2650         2.277               239           0.008                12  ...   
2651         2.260               239           0.010                12  ...   
2652         2.254               239           0.007                12  ...   

      defects                                                          \
     bubble_1 exfoliation_1 stain_1 dent_1 deformation_1 short_shot_2   
0           0             0       0      0             0            0   
1           0             0       0      0             0            0   
2           0             0       0      0             0            0   
3           0             1       0      0             0            0   
4           0             0       0      0             0            0   
...       ...           ...     ...    ...           ...          ...   
2648        0             0       0      0             0            0   
2649        0             0       0      0             0            0   
2650        0             0       0      0             0            0   
2651        0             0       0      0             0            0   
2652        0             0       0      0             0            0   

                                          defect_flag  
     bubble_2 exfoliation_2 deformation_2   is_defect  
0           0             0             0           0  
1           0             0             0           0  
2           0             0             0           0  
3           0             0             0           1  
4           0             0             0           0  
...       ...           ...           ...         ...  
2648        0             0             0           0  
2649        0             0             0           0  
2650        0             0             0           0  
2651        0             0             0           0  
2652        0             0             0           0  

[2653 rows x 42 columns]

In [15]:
# Product_Type이 1인 행만 필터링하여 저장
d_reduced.to_csv('product_type_1.csv', index=False, encoding='utf-8-sig')

In [16]:
df_type_2 = df_cleaned[df_cleaned[('process', 'product_type')] == 2].reset_index(drop=True)

In [18]:
defect_cols = [col for col in df_type_2.columns if 'defect' in str(col)]

for col in defect_cols:
    print(f"--- [Column: {col}] ---")
    print(df_type_2[col].value_counts())
    print("\n") 

# 각 컬럼의 value_counts를 데이터프레임 하나로 병합
counts_summary = df_type_2[defect_cols].apply(lambda x: x.value_counts()).fillna(0).astype(int)
display(counts_summary)

--- [Column: ('defects', 'short_shot_1')] ---
(defects, short_shot_1)
0    1788
1     176
Name: count, dtype: int64


--- [Column: ('defects', 'bubble_1')] ---
(defects, bubble_1)
0    1963
1       1
Name: count, dtype: int64


--- [Column: ('defects', 'exfoliation_1')] ---
(defects, exfoliation_1)
0    1964
Name: count, dtype: int64


--- [Column: ('defects', 'blow_hole_1')] ---
(defects, blow_hole_1)
0    1852
1     112
Name: count, dtype: int64


--- [Column: ('defects', 'stain_1')] ---
(defects, stain_1)
0    1871
1      93
Name: count, dtype: int64


--- [Column: ('defects', 'dent_1')] ---
(defects, dent_1)
0    1960
1       4
Name: count, dtype: int64


--- [Column: ('defects', 'deformation_1')] ---
(defects, deformation_1)
0    1964
Name: count, dtype: int64


--- [Column: ('defects', 'contamination_1')] ---
(defects, contamination_1)
0    1960
1       4
Name: count, dtype: int64


--- [Column: ('defects', 'impurity_1')] ---
(defects, impurity_1)
0    1962
1       2
Name: count,

defects                                                    \
  short_shot_1 bubble_1 exfoliation_1 blow_hole_1 stain_1 dent_1   
0         1788     1963          1964        1852    1871   1960   
1          176        1             0         112      93      4   

                                                    ...                 \
  deformation_1 contamination_1 impurity_1 crack_1  ... stain_2 dent_2   
0          1964            1960       1962    1963  ...    1964   1960   
1             0               4          2       1  ...       0      4   

                                                                            \
  deformation_2 contamination_2 impurity_2 crack_2 scratch_2 buring_mark_2   
0          1964            1956       1959    1962      1964          1964   
1             0               8          5       2         0             0   

               defect_flag  
  inclusions_2   is_defect  
0         1963        1467  
1            1         497  

[2 rows x 27 columns]

In [19]:
# 값이 모두 0인 컬럼명 리스트 생성
only_zero_cols = [col for col in defect_cols if (df_type_2[col] == 0).all()]

print(f"값이 0만 있는 컬럼들({len(only_zero_cols)}개): \n{only_zero_cols} ")

값이 0만 있는 컬럼들(9개): 
[('defects', 'exfoliation_1'), ('defects', 'deformation_1'), ('defects', 'inclusions_1'), ('defects', 'bubble_2'), ('defects', 'exfoliation_2'), ('defects', 'stain_2'), ('defects', 'deformation_2'), ('defects', 'scratch_2'), ('defects', 'buring_mark_2')] 


In [20]:
d_reduced = df_type_2.drop(columns=only_zero_cols)

d_reduced

process                                                     \
           id product_type shot velocity_1 velocity_2 velocity_3   
0     4207011            2   11      0.156      0.166      0.192   
1     4208012            2   12      0.157      0.166      0.204   
2     4209013            2   13      0.156      0.170      0.204   
3     4210014            2   14      0.154      0.170      0.202   
4     4211015            2   15      0.146      0.160      0.198   
...       ...          ...  ...        ...        ...        ...   
1959  7525657            2  657      0.144      0.173      0.200   
1960  7527658            2  658      0.144      0.173      0.200   
1961  7529659            2  659      0.150      0.166      0.210   
1962  7531660            2  660      0.144      0.174      0.206   
1963  7533661            2  661      0.147      0.174      0.204   

                                                                        ...  \
     high_velocity cylinder_pressure rapid_rise_time biscuit_thickness  ...   
0            2.723               265           0.012                20  ...   
1            2.730               264           0.014                19  ...   
2            2.715               265           0.012                18  ...   
3            2.717               264           0.011                20  ...   
4            2.684               264           0.012                20  ...   
...            ...               ...             ...               ...  ...   
1959         2.536               264           0.012                17  ...   
1960         2.536               264           0.012                17  ...   
1961         2.492               265           0.011                17  ...   
1962         2.514               264           0.011                16  ...   
1963         2.532               265           0.012                18  ...   

       defects                                                                \
     scratch_1 buring_mark_1 short_shot_2 blow_hole_2 dent_2 contamination_2   
0            0             0            0           0      0               0   
1            0             0            0           0      0               0   
2            0             0            0           0      0               0   
3            0             0            0           0      0               0   
4            0             0            0           0      0               0   
...        ...           ...          ...         ...    ...             ...   
1959         0             0            0           0      0               0   
1960         0             0            0           0      0               0   
1961         0             0            0           0      0               0   
1962         0             0            0           0      0               0   
1963         0             0            0           0      0               0   

                                     defect_flag  
     impurity_2 crack_2 inclusions_2   is_defect  
0             0       0            0           0  
1             0       0            0           0  
2             0       0            0           0  
3             0       0            0           0  
4             0       0            0           0  
...         ...     ...          ...         ...  
1959          0       0            0           0  
1960          0       0            0           0  
1961          0       0            0           0  
1962          0       0            0           1  
1963          0       0            0           1  

[1964 rows x 49 columns]

In [21]:
# 2. Product_Type이 2인 행만 필터링하여 저장
df_type_2.to_csv('product_type_2.csv', index=False, encoding='utf-8-sig')

In [22]:
df1 = pd.read_csv('product_type_1.csv',header=[0,1])
df2 = pd.read_csv('product_type_2.csv',header=[0,1])

In [23]:
# 컬럼별 결측치 개수 합계
print(df1.isnull().sum())

process      id                      0
             product_type            0
             shot                    0
             velocity_1              0
             velocity_2              0
             velocity_3              0
             high_velocity           0
             cylinder_pressure       0
             rapid_rise_time         0
             biscuit_thickness       0
             clamping_force          0
             cycle_time              0
             pressure_rise_time      0
             casting_pressure        0
             spray_time              0
             spray_1_time            0
             spray_2_time            0
sensor       melting_furnace_temp    0
             air_pressure            0
             air_pressure_min        0
             air_pressure_max        0
             coolant_temp            0
             coolant_temp_min        0
             coolant_temp_max        0
             coolant_pressure        0
             factory_temp

In [24]:
# 컬럼별 결측치 개수 합계
print(df2.isnull().sum())

process      id                       0
             product_type             0
             shot                     0
             velocity_1               0
             velocity_2               0
             velocity_3               0
             high_velocity            0
             cylinder_pressure        0
             rapid_rise_time          0
             biscuit_thickness        0
             clamping_force           0
             cycle_time               0
             pressure_rise_time       0
             casting_pressure         0
             spray_time               0
             spray_1_time             0
             spray_2_time             0
sensor       melting_furnace_temp     0
             air_pressure             0
             air_pressure_min         0
             air_pressure_max         0
             coolant_temp             0
             coolant_temp_min         0
             coolant_temp_max         0
             coolant_pressure         0


In [25]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2653 entries, 0 to 2652
Data columns (total 42 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   (process, id)                   2653 non-null   int64  
 1   (process, product_type)         2653 non-null   int64  
 2   (process, shot)                 2653 non-null   int64  
 3   (process, velocity_1)           2653 non-null   float64
 4   (process, velocity_2)           2653 non-null   float64
 5   (process, velocity_3)           2653 non-null   float64
 6   (process, high_velocity)        2653 non-null   float64
 7   (process, cylinder_pressure)    2653 non-null   int64  
 8   (process, rapid_rise_time)      2653 non-null   float64
 9   (process, biscuit_thickness)    2653 non-null   int64  
 10  (process, clamping_force)       2653 non-null   int64  
 11  (process, cycle_time)           2653 non-null   float64
 12  (process, pressure_rise_time)   2653 non-null

In [26]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1964 entries, 0 to 1963
Data columns (total 58 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   (process, id)                   1964 non-null   int64  
 1   (process, product_type)         1964 non-null   int64  
 2   (process, shot)                 1964 non-null   int64  
 3   (process, velocity_1)           1964 non-null   float64
 4   (process, velocity_2)           1964 non-null   float64
 5   (process, velocity_3)           1964 non-null   float64
 6   (process, high_velocity)        1964 non-null   float64
 7   (process, cylinder_pressure)    1964 non-null   int64  
 8   (process, rapid_rise_time)      1964 non-null   float64
 9   (process, biscuit_thickness)    1964 non-null   int64  
 10  (process, clamping_force)       1964 non-null   int64  
 11  (process, cycle_time)           1964 non-null   float64
 12  (process, pressure_rise_time)   1964 non-null